# Netflix Recommendation System — Improved
Key improvements over v1:
- Uses more data (up to 500k rows) for better signal
- GridSearchCV to tune SVD hyperparameters
- Fixed Precision@K formula (divide by K, not predicted_relevant_count)
- Added MAE, F1@K, and NDCG@K metrics
- Reports results in a clean summary table

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
!pip install scikit-surprise --quiet

## 1. Load and Parse Data

In [ ]:
df = pd.read_csv(
    '/content/combined_data_1.txt',
    header=None,
    usecols=[0, 1],
    names=['CustID', 'Ratings']
)

# Build movie_id column
movie_id = []
current_movie = None
for x in df['CustID']:
    if ':' in str(x):
        current_movie = int(str(x).replace(':', ''))
    movie_id.append(current_movie)

df['movie_id'] = movie_id
df.dropna(inplace=True)
df['CustID'] = df['CustID'].astype(int)
df['movie_id'] = df['movie_id'].astype(int)
df['Ratings'] = df['Ratings'].astype(int)

print(f'Total ratings: {len(df):,}')
df.head()

## 2. Filter Sparse Users and Movies

In [ ]:
# Remove users who rated fewer movies than the 60th percentile
user_counts = df.groupby('CustID')['Ratings'].count()
user_bench = round(user_counts.quantile(0.60))
rejected_users = user_counts[user_counts < user_bench].index
df = df[~df['CustID'].isin(rejected_users)]

# Remove movies rated by fewer people than the 60th percentile
movie_counts = df.groupby('movie_id')['Ratings'].count()
movie_bench = round(movie_counts.quantile(0.60))
rejected_movies = movie_counts[movie_counts < movie_bench].index
df = df[~df['movie_id'].isin(rejected_movies)]

print(f'Ratings after filtering: {len(df):,}')
print(f'Unique users: {df["CustID"].nunique():,}')
print(f'Unique movies: {df["movie_id"].nunique():,}')

## 3. Rating Distribution

In [ ]:
plt.figure(figsize=(7, 4))
df['Ratings'].value_counts().sort_index().plot(kind='bar', color='steelblue', edgecolor='white')
plt.title('Rating Distribution')
plt.xlabel('Rating')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Prepare Data for Surprise

**Improvement:** Using up to 500k rows instead of 100k so the model gets far more signal.

In [ ]:
from surprise import Reader, Dataset, SVD
from surprise.model_selection import cross_validate, train_test_split, GridSearchCV

# Use up to 500k rows — significantly more signal than 100k
sample_size = min(500_000, len(df))
df_sample = df.sample(n=sample_size, random_state=42)

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df_sample[['CustID', 'movie_id', 'Ratings']], reader)
print(f'Dataset loaded with {sample_size:,} ratings')

## 5. Hyperparameter Tuning with GridSearchCV

**Improvement:** Default SVD params are rarely optimal. We search over key hyperparameters.

In [ ]:
param_grid = {
    'n_factors': [50, 100, 150],   # latent dimensions
    'n_epochs': [20, 30],           # training epochs
    'lr_all': [0.002, 0.005],       # learning rate
    'reg_all': [0.02, 0.1]          # regularization (prevents overfitting)
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs.fit(data)

best_params = gs.best_params['rmse']
print('Best RMSE:', round(gs.best_score['rmse'], 4))
print('Best MAE: ', round(gs.best_score['mae'], 4))
print('Best params:', best_params)

## 6. Train Final Model with Best Parameters

In [ ]:
train, test = train_test_split(data, test_size=0.25, random_state=42)

best_model = SVD(**best_params)
best_model.fit(train)
predictions = best_model.test(test)

print(f'Trained on {len(train.ur):,} users, testing on {len(test):,} predictions')

## 7. Evaluation: Precision@K, Recall@K, F1@K, NDCG@K

**Bug fix in v1:** Precision@K should divide by K (the number of recommendations), not by the count of predicted-relevant items in top-K. The original formula inflated precision artificially.

**New metrics added:** F1@K (harmonic mean of P and R), NDCG@K (rewards correct items ranked higher).

In [ ]:
from collections import defaultdict

def precision_recall_at_k(predictions, k=10, threshold=3.5):
    """Compute Precision@K, Recall@K, F1@K per user then average.
    
    Fix vs v1: Precision@K = hits / k  (not hits / predicted_relevant_count)
    The standard definition asks: of the K items we recommended,
    how many did the user actually like?
    """
    user_est_true = defaultdict(list)
    for uid, _, r_ui, est, _ in predictions:
        user_est_true[uid].append((est, r_ui))

    precisions, recalls, f1s = [], [], []

    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = user_ratings[:k]

        # Total truly relevant items across all user ratings
        relevant_count = sum(1 for _, true_r in user_ratings if true_r >= threshold)

        # Hits: items that are both in top-K and truly relevant
        hits = sum(1 for est, true_r in top_k if true_r >= threshold)

        # Precision@K: correct / K  (FIXED from v1)
        prec = hits / k
        precisions.append(prec)

        # Recall@K: correct / total relevant
        rec = (hits / relevant_count) if relevant_count > 0 else 0
        recalls.append(rec)

        # F1@K
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0
        f1s.append(f1)

    return (
        sum(precisions) / len(precisions),
        sum(recalls) / len(recalls),
        sum(f1s) / len(f1s)
    )


def ndcg_at_k(predictions, k=10, threshold=3.5):
    """NDCG@K — rewards relevant items ranked higher in the list."""
    user_est_true = defaultdict(list)
    for uid, _, r_ui, est, _ in predictions:
        user_est_true[uid].append((est, r_ui))

    ndcgs = []
    for uid, user_ratings in user_est_true.items():
        user_ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = user_ratings[:k]

        # DCG: relevance / log2(rank+1)
        dcg = sum(
            (1 if true_r >= threshold else 0) / np.log2(rank + 2)
            for rank, (_, true_r) in enumerate(top_k)
        )
        # Ideal DCG: all relevant items at the top
        n_relevant = sum(1 for _, true_r in user_ratings if true_r >= threshold)
        ideal_k = min(k, n_relevant)
        idcg = sum(1 / np.log2(rank + 2) for rank in range(ideal_k))

        ndcgs.append(dcg / idcg if idcg > 0 else 0)

    return sum(ndcgs) / len(ndcgs)

In [ ]:
K = 10
THRESHOLD = 3.5

precision, recall, f1 = precision_recall_at_k(predictions, k=K, threshold=THRESHOLD)
ndcg = ndcg_at_k(predictions, k=K, threshold=THRESHOLD)

results = pd.DataFrame({
    'Metric': [f'Precision@{K}', f'Recall@{K}', f'F1@{K}', f'NDCG@{K}'],
    'Score': [precision, recall, f1, ndcg]
})
results['Score'] = results['Score'].round(4)
print(results.to_string(index=False))

## 8. Visualise Metrics

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']
bars = ax.bar(results['Metric'], results['Score'], color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, results['Score']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=11)
ax.set_ylim(0, 1.0)
ax.set_title(f'Recommender Metrics @ K={K}')
ax.set_ylabel('Score')
plt.tight_layout()
plt.show()

## 9. Generate Recommendations for a User

In [ ]:
df_title = pd.read_csv(
    '/content/movie_titles.csv',
    header=None,
    usecols=[0, 1, 2],
    names=['movie_id', 'year', 'name'],
    encoding='latin'
)

TARGET_USER = 1333

recs = df_title.copy()
recs['estimated_score'] = recs['movie_id'].apply(
    lambda x: best_model.predict(TARGET_USER, x).est
)

top10 = recs.sort_values('estimated_score', ascending=False).head(10)
print(f'Top 10 recommendations for user {TARGET_USER}:')
top10[['name', 'year', 'estimated_score']].reset_index(drop=True)

## 10. Summary of Improvements

| Change | v1 | v2 (this notebook) |
|---|---|---|
| Data size | 100k rows | up to 500k rows |
| Hyperparameter tuning | None (defaults) | GridSearchCV over n_factors, n_epochs, lr_all, reg_all |
| Precision@K formula | hits / predicted_relevant (inflated) | hits / K (correct) |
| Metrics reported | P@K, R@K | P@K, R@K, F1@K, NDCG@K |
| rating_scale set | No | Yes (1–5) |
| random_state | Not set | 42 (reproducible splits) |